# Shared SSL Phase 1: SimMTM Pre-Training

## Overview
This notebook implements the **Self-Supervised Learning (SSL)** pre-training for the **SimMTM** architecture using the masked patch reconstruction task. 
This is a **shared step**: the encoder is pre-trained once on Corponi unlabeled data to learn robust signal features. The resulting weights are then used for supervised fine-tuning on both **Hosseini** and **WESAD** datasets.

### Instructions:
1. Run this notebook to generate `simmtm_encoder_pretrained.pt`.
2. Use the generated weights in the comparison notebooks found in the dataset-specific folders.

In [ ]:
# ── CELL 0: Setup and Dependencies ──────────────────────────────────────
import os, sys, shutil, glob, h5py, pickle
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
from einops import rearrange
from torch.utils.data import Dataset, DataLoader

WORK_DIR = '/kaggle/working/e4selflearning'
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

!pip install -q biosppy flirt ruamel.yaml hrv-analysis neurokit2 distinctipy joypy peakutils biopsykit

if WORK_DIR not in sys.path: sys.path.insert(0, f"{WORK_DIR}/src")
os.chdir(WORK_DIR)

### 1. Dataset Ingestion
Preparing segments for the masked patches task.

In [ ]:
import os, sys, glob, shutil, pickle

WORK_DIR = '/kaggle/working/e4selflearning'
if WORK_DIR not in sys.path: sys.path.insert(0, f"{WORK_DIR}/src")
os.chdir(WORK_DIR)

# --- PATCH 1: Numpy 2.0 (newshape -> shape) ---
def apply_patch(file_path, old_str, new_str):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f: c = f.read()
        if old_str in c:
            c = c.replace(old_str, new_str)
            with open(file_path, 'w') as f: f.write(c)
            print(f"Patch OK: {file_path}")

apply_patch('segment.py', 'newshape=', 'shape=')
apply_patch('src/timebase/data/preprocessing.py', 'newshape=', 'shape=')

# --- PATCH 2: Aggiunge 'dati_preelaborati' a static.py ---
static_p = 'src/timebase/data/static.py'
if os.path.exists(static_p):
    with open(static_p, 'r') as f: c = f.read()
    if '"dati_preelaborati"' not in c:
        c = c.replace('    "wesd": 12,\n}', '    "wesd": 12,\n    "dati_preelaborati": 13,\n}')
        c = c.replace('    "wesd": "lawngreen",\n}', '    "wesd": "lawngreen",\n    "dati_preelaborati": "steelblue",\n}')
        with open(static_p, 'w') as f: f.write(c)
        print("Patch OK: static.py (dati_preelaborati)")

TARGET_DIR = '/kaggle/working/e4selflearning/data/preprocessed/unsegmented_unlabelled'
if os.path.exists(TARGET_DIR): shutil.rmtree(TARGET_DIR)
os.makedirs(TARGET_DIR, exist_ok=True)

print("Ricerca dati nel pannello Input...")
meta_files = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)
if not meta_files: 
    raise FileNotFoundError("ERRORE: metadata.pkl non trovato!")
src_dir = os.path.dirname(meta_files[0])

for item in os.listdir(src_dir):
    s, d = os.path.join(src_dir, item), os.path.join(TARGET_DIR, item)
    if item == 'metadata.pkl': shutil.copy2(s, d) 
    else: os.symlink(s, d)

with open(os.path.join(TARGET_DIR, 'metadata.pkl'), 'rb') as f: meta = pickle.load(f)
new_sessions = {}
for root, dirs, files in os.walk(TARGET_DIR, followlinks=True):
    if 'channels.h5' in files:
        clean_name = os.path.basename(root)
        new_rel_id = os.path.relpath(root, TARGET_DIR).replace('\\', '/')
        for old_id in meta['sessions_info'].keys():
            if old_id.endswith(clean_name):
                new_sessions[new_rel_id] = meta['sessions_info'][old_id]; break

meta['sessions_info'] = new_sessions
with open(os.path.join(TARGET_DIR, 'metadata.pkl'), 'wb') as f: pickle.dump(meta, f)
print("Ingestion completata!")

print("\nInizio Segmentazione...")
!PYTHONPATH=src python segment.py --data_dir {TARGET_DIR} --output_dir data/preprocessed/sl640_ss640_unlabelled --segment_length 640 --segmentation_mode 1 --step_size 640 --num_workers 2 --overwrite


### 2. SimMTM Architecture

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, patch_size, d_model):
        super().__init__()
        self.proj = nn.Linear(patch_size, d_model); self.norm = nn.LayerNorm(d_model)
    def forward(self, x): return self.norm(self.proj(x))

class SimMTMEncoder(nn.Module):
    def __init__(self, seg_len=640, patch_size=16, d_model=128, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.patch_size, self.d_model = patch_size, d_model
        self.n_patches = seg_len // patch_size
        self.patch_emb = PatchEmbedding(patch_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers); self.norm = nn.LayerNorm(d_model)
    
    def patchify(self, x):
        B, C, T = x.shape
        return rearrange(x[:, :, :self.n_patches * self.patch_size], 'b c (n p) -> (b c) n p', p=self.patch_size)

    def forward(self, x):
        patches = self.patchify(x)
        emb = self.patch_emb(patches) + self.pos_emb
        return self.norm(self.transformer(emb))

### 3. Masked SSL Training

In [ ]:
CHANNELS = ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
class SSLDataset(Dataset):
    def __init__(self, h5_list): self.files = h5_list
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        with h5py.File(self.files[i], 'r') as f:
            sigs = [F.interpolate(torch.tensor(f[c][:]).float().view(1,1,-1), size=640, mode='linear').squeeze() for c in CHANNELS]
            sigs = [(s - s.mean())/(s.std()+1e-8) for s in sigs]
            return torch.stack(sigs)

h5_files = [os.path.join(WORK_DIR, p) for p in pickle.load(open('data/preprocessed/sl640_ss640_unlabelled/metadata.pkl', 'rb'))['sessions_paths']]
loader = DataLoader(SSLDataset(h5_files), batch_size=32, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = SimMTMEncoder().to(device); decoder = nn.Linear(128, 16).to(device)
optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-4)

print("Starting SimMTM SSL Pre-training (20 epochs)...")
for epoch in range(1, 21):
    encoder.train(); decoder.train(); total_loss = 0.0
    for xb in loader:
        xb = xb.to(device); B_C = xb.shape[0]*xb.shape[1]
        patches = encoder.patchify(xb)
        mask = torch.rand(B_C, encoder.n_patches, device=device) < 0.4
        p_masked = patches.clone(); p_masked[mask] = 0.0
        enc = encoder.norm(encoder.transformer(encoder.patch_emb(p_masked) + encoder.pos_emb))
        loss = F.mse_loss(decoder(enc)[mask], patches[mask])
        optimizer.zero_grad(); loss.backward(); optimizer.step(); total_loss += loss.item()
    print(f"Epoch {epoch:2d}/20 | Loss (MSE): {total_loss/len(loader):.4f}")

torch.save(encoder.state_dict(), '/kaggle/working/simmtm_encoder_pretrained.pt')
print("SimMTM Pre-trained weights saved.")